###### Nemotron 3 Nano — SDPG / Dr.GRPO RLVR (v19)

GRPO with **verifiable rewards (RLVR)**, optional **SDPG** self-distillation.
Built to **resume on top of the 85% SFT adapter**.

- **Data** (`DATA_SOURCE`): real competition `train.csv` (prompt+answer, matches
  the eval distribution — recommended for resuming), and/or the
  `RLVR/synthetic_problems.py` generators. 6 puzzle types.
- **Prompt format = eval format.** Eval gives the model the **bare** problem
  (no per-category system prompt); the model self-identifies the type. We train
  with one fixed wrapper (`EVAL_INSTRUCTION`), identical for all types
  (CLAUDE.md gotcha #10). The category is used **only** server-side to pick the
  verifier — never injected into the prompt.
- **Reward**: type-aware `\boxed{}` verifier + `<think>`-before-boxed format +
  DAPO overlong shaping.
- **Core RL**: Dr.GRPO (arXiv:2503.20783) — `loss_type=dr_grpo`,
  `scale_rewards=False`, LR 1e-6 const, small ref-KL anchor to the SFT.
- **SDPG** (optional): teacher = the same policy conditioned on the privileged
  answer; full-vocab reverse-KL `D_KL(π_θ(·|x)‖π_θ(·|c,x))` densifies the sparse
  verifier reward — useful at 85% where reward saturates.
  (Liu·Zhang·Zhang·Gu, `github.com/lauyikfung/SDPG`.)

| SDPG term | This notebook |
|---|---|
| GRPO clipped advantage (group-relative, std-norm) | `GRPOTrainer` core (`loss_type=dr_grpo`) |
| Reference-policy KL (α=1e-3) | `GRPOConfig(beta=SDPG_ALPHA_REF)` — anchors near the SFT |
| Self-distillation reverse-KL (β=1e-3) | `SDPGTrainer.compute_loss` adds `SDPG_BETA·KL(student‖teacher)` |


In [ ]:
import os, sys
os.environ["PYTHONIOENCODING"] = "utf-8"
if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8", errors="replace")
if hasattr(sys.stderr, "reconfigure"):
    sys.stderr.reconfigure(encoding="utf-8", errors="replace")

# ── Run mode ─────────────────────────────────────────────────────────────────
TRAIN_ON_KAGGLE = 1          # train LoRA on the Kaggle/RTX-6000 GPU
SEED            = 3407
MODEL_MAX_LEN   = 7168        # base context (v18 parity)

# ── Warm-start from the 0.85 SFT adapter (EXACT v18 resolution) ──────────────
# Resolved in order: Kaggle MODEL slug -> dataset/model dir -> auto-glob.
# Kaggle Model slug = "<user>/<model>/<framework>/<variation>".
SFT_ADAPTER_MODEL_SLUG = ""   # "" -> skip kagglehub; use SFT_ADAPTER_DIR below
SFT_ADAPTER_DIR        = "/kaggle/input/models/ramkan07/nemotron-lora-adaptor/pytorch/default/1"

# ── Data source for RL ───────────────────────────────────────────────────────
# "train_csv": real competition train.csv (matches eval). RECOMMENDED.
# "synthetic": RLVR generators only.   "mixed": both.
DATA_SOURCE    = "train_csv"
TRAIN_CSV_PATH = ("/kaggle/input/competitions/nvidia-nemotron-model-reasoning-challenge/train.csv"
                  if TRAIN_ON_KAGGLE else "../data_generation/src/train.csv")
MAX_TRAIN_ROWS = 4000        # cap real rows per run (None = all)
SYNTH_PER_CAT  = 200         # synthetic problems per category (synthetic/mixed)

# ── Prompt format — MUST match eval ──────────────────────────────────────────
# Eval feeds the BARE problem text; the model self-identifies the puzzle type.
USE_CATEGORY_SYSTEM_PROMPT = 0          # keep 0 — category prompts = train/eval mismatch
EVAL_INSTRUCTION = "\nPlease put your final answer inside \\boxed{}."  # "" if eval passes raw prompt
GENERIC_SYSTEM_PROMPT = ""               # single system msg for ALL problems ("" = none)

# ── SDPG switch ──────────────────────────────────────────────────────────────
# 0 -> plain Dr.GRPO (safe).  1 -> add self-distillation (2 extra full-vocab
# forwards; RTX 6000 Pro / A100-80G+). NOTE: SDPG teacher uses LEFT-padded
# prompts; Nemotron Mamba-2 can NaN on left-pad batches -> keep 0 unless verified.
USE_SDPG       = 0
SDPG_BETA          = 1e-3
SDPG_ALPHA_REF     = 0.0    # beta=0 Dr.GRPO: NO reference forward (halves per-step compute + a full-vocab logit spike)
SDPG_DISTILL_MAX_T = 512
SDPG_DISTILL_CHUNK = 128

print(f"TRAIN_ON_KAGGLE={TRAIN_ON_KAGGLE}  USE_SDPG={USE_SDPG}  SEED={SEED}")
print(f"DATA_SOURCE={DATA_SOURCE}  SFT_ADAPTER_DIR={SFT_ADAPTER_DIR}")
print(f"USE_CATEGORY_SYSTEM_PROMPT={USE_CATEGORY_SYSTEM_PROMPT}  EVAL_INSTRUCTION={EVAL_INSTRUCTION!r}")


In [ ]:
# Triton wheel — bundled in a Kaggle dataset (no internet at eval time).
if TRAIN_ON_KAGGLE:
    import glob, subprocess, site
    candidates = glob.glob("/kaggle/input/**/*triton*.whl", recursive=True)
    print("Triton wheels:", candidates)
    if candidates:
        target = "/kaggle/working/pydeps"
        os.makedirs(target, exist_ok=True)
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "--no-deps", "--target", target,
             "--upgrade", "--ignore-installed", candidates[0]],
            check=True,
        )
        if target not in sys.path:
            sys.path.insert(0, target)
        site.addsitedir(target)
    else:
        print("No Triton wheel found — assuming the env already provides Triton.")


In [ ]:
# ptxas for Blackwell (RTX 6000 Pro) — copy the bundled utility-script binary.
if TRAIN_ON_KAGGLE:
    import shutil, stat
    sys.path.insert(0, "/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script")
    ptxas_src = ("/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script/"
                 "triton/backends/nvidia/bin/ptxas-blackwell")
    ptxas_dst = "/tmp/ptxas-blackwell"
    if os.path.exists(ptxas_src) and not os.path.exists(ptxas_dst):
        shutil.copy2(ptxas_src, ptxas_dst)
        os.chmod(ptxas_dst, os.stat(ptxas_dst).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)
        os.environ["TRITON_PTXAS_PATH"] = ptxas_dst
        print("ptxas-blackwell ready ->", ptxas_dst)
    else:
        print("ptxas shim skipped (src missing or already present).")


In [ ]:
BASE_MODEL_NAME = "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16"

OUTPUT_ROOT    = "/kaggle/working/outputs/v19_sdpg" if TRAIN_ON_KAGGLE else "outputs/v19_sdpg"
GRPO_ADAPTER_DIR = os.path.join(OUTPUT_ROOT, "grpo_adapter")
SUBMISSION_DIR   = os.path.join(OUTPUT_ROOT, "submission")
TB_LOG_DIR       = os.path.join(OUTPUT_ROOT, "tb")
for d in (OUTPUT_ROOT, GRPO_ADAPTER_DIR, SUBMISSION_DIR, TB_LOG_DIR):
    os.makedirs(d, exist_ok=True)

MAX_LORA_RANK = 32           # submission constraint (CLAUDE.md — non-negotiable)
MAX_SEQ_LEN   = MODEL_MAX_LEN
print("Output root:", OUTPUT_ROOT)


In [ ]:
# Offline package install (EXACT v18 path) — find-links dir + mamba/causal wheels.
if TRAIN_ON_KAGGLE:
    import glob, os, subprocess, sys

    def recursive_wheels(pattern: str):
        return sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))

    packages_dir = "/kaggle/input/datasets/mayukh18/nemotron-packages/packages"
    all_mamba  = recursive_wheels("mamba_ssm-*.whl")
    all_causal = recursive_wheels("causal*conv1d*.whl")

    import torch
    print("Torch:", torch.__version__, "CUDA:", torch.cuda.is_available())
    if not torch.cuda.is_available():
        raise RuntimeError("TRAIN_ON_KAGGLE=1 requires a GPU runtime.")
    if not os.path.isdir(packages_dir):
        raise FileNotFoundError(f"Offline wheel directory not found: {packages_dir}")

    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "--no-index",
         "--find-links", packages_dir, "unsloth", "trl", "peft", "transformers",
         "datasets", "accelerate", "bitsandbytes"],
        check=True,
    )

    def pick_last(w): return w[-1] if w else None
    causal_wheel = pick_last(all_causal)
    mamba_wheel  = pick_last(all_mamba)
    if causal_wheel:
        subprocess.run([sys.executable, "-m", "pip", "install", "--no-index", "--no-deps", causal_wheel], check=True)
    if mamba_wheel:
        subprocess.run([sys.executable, "-m", "pip", "install", "--no-index", "--no-deps", mamba_wheel], check=True)
    else:
        raise FileNotFoundError("No compatible mamba_ssm wheel found under /kaggle/input.")
    print("Offline package installation finished.")


In [ ]:
if TRAIN_ON_KAGGLE:
    import torch
    import kagglehub
    from unsloth import FastLanguageModel

    MODEL_PATH = kagglehub.model_download("metric/nemotron-3-nano-30b-a3b-bf16/transformers/default")
    print(f"Model path: {MODEL_PATH}")

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_PATH,
        max_seq_length=MODEL_MAX_LEN,
        load_in_4bit=False,
        load_in_8bit=False,
        full_finetuning=False,
        trust_remote_code=True,
        unsloth_force_compile=False,
        attn_implementation="eager",
        dtype=torch.bfloat16,
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    # left padding for batched GENERATION; TRL GRPO manages its own padding.
    tokenizer.padding_side = "left"
    print("Model loaded with Unsloth.")
else:
    print("TRAIN_ON_KAGGLE=0: skipping base model and tokenizer loading.")


## LoRA: warm-start from SFT adapter (v18-exact loadup)

Resolves the adapter via `SFT_ADAPTER_MODEL_SLUG` (Kaggle Models) -> `SFT_ADAPTER_DIR` -> auto-glob, then `PeftModel.from_pretrained(is_trainable=True)`. Cold-start fallback = RSLoRA `r=32, alpha=64` on the regex targets (`self_attn q/k/v/o`, `mamba in/out/x/dt/gate`, `shared_experts gate/up/down`). Grad-path hardening + a trainable-param audit, then a non-fatal `r<=32` submission check.

In [ ]:
from peft import PeftModel, LoraConfig, get_peft_model, TaskType
import re, os, glob

# ── Resolve the warm-start adapter dir (Kaggle MODEL slug -> dataset dir -> glob) ──
def _resolve_adapter_dir():
    if SFT_ADAPTER_MODEL_SLUG:
        try:
            import kagglehub
            p = kagglehub.model_download(SFT_ADAPTER_MODEL_SLUG)
            print(f"[adapter] kagglehub -> {p}")
            if os.path.exists(os.path.join(p, "adapter_config.json")):
                return p
            hits = glob.glob(os.path.join(p, "**", "adapter_config.json"), recursive=True)
            if hits:
                return os.path.dirname(sorted(hits, key=len)[0])
        except Exception as e:
            print(f"[adapter] kagglehub failed ({type(e).__name__}: {e}); trying dataset path")
    if os.path.exists(os.path.join(SFT_ADAPTER_DIR, "adapter_config.json")):
        return SFT_ADAPTER_DIR
    hits = glob.glob("/kaggle/input/**/adapter_config.json", recursive=True)
    if hits:
        d = os.path.dirname(sorted(hits, key=len)[0])
        print(f"[adapter] auto-found -> {d}")
        return d
    return None

ADAPTER_DIR = _resolve_adapter_dir()

# Inventory linear modules (used only for the cold-start regex / debug).
linear_modules = []
for name, mod in model.named_modules():
    if mod.__class__.__name__ in ("Linear", "Linear4bit", "Linear8bitLt"):
        linear_modules.append(name)

target_regex = (
    r".*("
    r"self_attn\.(q|k|v|o)_proj"
    r"|mamba\.(in|out|x|dt|gate)_proj"
    r"|shared_experts\.(gate|up|down)_proj"
    r")$"
)

if ADAPTER_DIR and os.path.exists(os.path.join(ADAPTER_DIR, "adapter_config.json")):
    print(f"[branch A: WARM-START] loading SFT adapter from {ADAPTER_DIR}")
    model = PeftModel.from_pretrained(model, ADAPTER_DIR, is_trainable=True)
else:
    print(f"[branch B: COLD-START] no adapter resolved "
          f"(slug={SFT_ADAPTER_MODEL_SLUG!r}, dir={SFT_ADAPTER_DIR!r}).")
    print(f"[branch B: COLD-START] fresh RSLoRA r=32 on base {BASE_MODEL_NAME}.")
    matched = [n for n in linear_modules if re.match(target_regex, n)]
    print(f"LoRA target regex matched {len(matched)} modules.")
    if len(matched) == 0:
        sample = [n for n in linear_modules if "expert" in n or "mamba" in n or "self_attn" in n][:20]
        raise RuntimeError(f"LoRA target_regex matched 0 modules. Sample: {sample}")
    lora_config = LoraConfig(
        r=32, lora_alpha=64, lora_dropout=0.0, bias="none",
        target_modules=target_regex, task_type=TaskType.CAUSAL_LM,
        use_rslora=True, use_dora=False,
    )
    model = get_peft_model(model, lora_config)

# ── grad-path hardening (real training vs dead zeros) ──
model.gradient_checkpointing_enable()
if hasattr(model, "enable_input_require_grads"):
    model.enable_input_require_grads()
try:
    model.config.use_cache = False
except Exception:
    pass
model.print_trainable_parameters()

trainable = [(n, p.numel()) for n, p in model.named_parameters() if p.requires_grad]
n_train = sum(s for _, s in trainable)
print(f"[audit] {len(trainable)} trainable tensors  total={n_train/1e6:.1f}M")
assert 30_000_000 <= n_train <= 1_200_000_000, \
    f"[audit] trainable {n_train/1e6:.1f}M out of expected range -- inspect adapter / regex."

# ── submission-constraint check (rank <= 32) — non-fatal, just warn ──
try:
    _ranks = [getattr(c, "r", None) for c in model.peft_config.values()]
    _maxr = max([r for r in _ranks if r] or [0])
    print("adapter rank(s):", _ranks)
    if _maxr > MAX_LORA_RANK:
        print(f"  *** WARNING: rank {_maxr} > {MAX_LORA_RANK} -> submission WILL be rejected. ***")
except Exception as e:
    print("rank check skipped:", repr(e))


## Synthetic data generators (vendored from `RLVR/synthetic_problems.py`)

Self-contained copies of the 6 generators — each returns `(prompt, answer)` with
a known ground truth, in the exact competition format.

In [ ]:
import random, string

CIPHER_SUBJECTS = ['alice','cat','rabbit','mouse','hatter','queen','king','knight',
    'princess','wizard','dragon','bird','turtle','student','teacher']
CIPHER_VERBS = ['follows','creates','draws','dreams','chases','reads','sees','watches',
    'explores','discovers','imagines','writes','studies','finds','found']
CIPHER_OBJECTS = ['castle','garden','book','mirror','key','map','door','forest','puzzle',
    'treasure','secret','tower','crystal','potion','message']
CIPHER_PLACES = ['wonderland','castle','forest','garden','valley','village','palace',
    'mountain','library','island','ocean','cave','school']
CIPHER_ADJECTIVES = ['the','magical','mysterious','golden','silver','bright','dark','clever',
    'colorful','hidden','ancient','strange','curious','wise']

def _random_phrase(rng, n_words=None):
    if n_words is None:
        n_words = rng.randint(2, 5)
    pool = CIPHER_SUBJECTS + CIPHER_VERBS + CIPHER_OBJECTS + CIPHER_PLACES + CIPHER_ADJECTIVES
    return " ".join(rng.choice(pool) for _ in range(n_words))

def _make_substitution_cipher(rng):
    letters = list(string.ascii_lowercase)
    shuffled = letters[:]
    rng.shuffle(shuffled)
    return dict(zip(letters, shuffled))

def _apply_cipher(text, cipher):
    return "".join(cipher.get(c, c) for c in text)

def generate_gravity_problem(rng, n_examples=5):
    g = round(rng.uniform(3.0, 30.0), 2)
    t_values = [round(rng.uniform(0.5, 6.0), 2) for _ in range(n_examples + 1)]
    examples, query_t = t_values[:n_examples], t_values[-1]
    lines = ["In Alice's Wonderland, the gravitational constant has been secretly changed. "
             "Here are some example observations:"]
    for t in examples:
        lines.append(f"For t = {t}s, distance = {round(0.5*g*t*t, 2)} m")
    lines.append(f"Now, determine the falling distance for t = {query_t}s given d = 0.5*g*t^2.")
    return "\n".join(lines), str(round(0.5*g*query_t*query_t, 2))

def generate_unit_conversion_problem(rng, n_examples=5):
    ratio = round(rng.uniform(0.3, 5.0), 4)
    inputs = [round(rng.uniform(1.0, 50.0), 2) for _ in range(n_examples + 1)]
    examples, query_inp = inputs[:n_examples], inputs[-1]
    lines = ["In Alice's Wonderland, a secret unit conversion is applied to measurements. For example:"]
    for inp in examples:
        lines.append(f"{inp} m becomes {round(inp*ratio, 2):.2f}")
    lines.append(f"Now, convert the following measurement: {query_inp} m")
    return "\n".join(lines), str(round(query_inp*ratio, 2))

_ROMAN = [(1000,"M"),(900,"CM"),(500,"D"),(400,"CD"),(100,"C"),(90,"XC"),
          (50,"L"),(40,"XL"),(10,"X"),(9,"IX"),(5,"V"),(4,"IV"),(1,"I")]
def _int_to_roman(n):
    out = []
    for val, sym in _ROMAN:
        while n >= val:
            out.append(sym); n -= val
    return "".join(out)

def generate_numeral_problem(rng, n_examples=4):
    used, nums = set(), []
    while len(nums) < n_examples:
        n = rng.randint(1, 3999)
        if n not in used:
            used.add(n); nums.append(n)
    q = rng.randint(1, 3999)
    while q in used:
        q = rng.randint(1, 3999)
    lines = ["In Alice's Wonderland, numbers are secretly converted into a different numeral system. "
             "Some examples are given below:"]
    for n in nums:
        lines.append(f"{n} -> {_int_to_roman(n)}")
    lines.append(f"Now, write the number {q} in the Wonderland numeral system.")
    return "\n".join(lines), _int_to_roman(q)

def generate_cipher_problem(rng, n_examples=5):
    cipher = _make_substitution_cipher(rng)
    phrases = [_random_phrase(rng, rng.randint(2, 5)) for _ in range(n_examples + 1)]
    examples, query = phrases[:n_examples], phrases[-1]
    lines = ["In Alice's Wonderland, secret encryption rules are used on text. Here are some examples:"]
    for p in examples:
        lines.append(f"{_apply_cipher(p, cipher)} -> {p}")
    lines.append(f"Now, decrypt the following text: {_apply_cipher(query, cipher)}")
    return "\n".join(lines), query

def _random_8bit_function(rng):
    t = rng.choice(["xor_mask","rol","ror","not_rol","xor_two_rols","per_bit_gate"])
    if t == "xor_mask":
        mask = [rng.randint(0,1) for _ in range(8)]
        return lambda bits, m=mask: [b ^ mi for b, mi in zip(bits, m)]
    if t == "rol":
        k = rng.randint(1,7); return lambda bits, k=k: bits[k:] + bits[:k]
    if t == "ror":
        k = rng.randint(1,7); return lambda bits, k=k: bits[-k:] + bits[:-k]
    if t == "not_rol":
        k = rng.randint(1,7); return lambda bits, k=k: [1-b for b in bits[k:] + bits[:k]]
    if t == "xor_two_rols":
        k1 = rng.randint(0,7); k2 = rng.randint(0,7)
        while k2 == k1:
            k2 = rng.randint(0,7)
        def fn(bits, k1=k1, k2=k2):
            a = bits[k1:] + bits[:k1]; b = bits[k2:] + bits[:k2]
            return [x ^ y for x, y in zip(a, b)]
        return fn
    bit_maps = []
    for _ in range(8):
        idx = [rng.randint(0,7) for _ in range(rng.randint(1,3))]
        bit_maps.append((idx, rng.choice(["and","or","xor","nand","nor"])))
    def per_bit(bits, bm=bit_maps):
        res = []
        for idx, gate in bm:
            vals = [bits[i] for i in idx]
            if gate == "and":
                v = 1
                for x in vals: v &= x
            elif gate == "or":
                v = 0
                for x in vals: v |= x
            elif gate == "xor":
                v = 0
                for x in vals: v ^= x
            elif gate == "nand":
                v = 1
                for x in vals: v &= x
                v = 1 - v
            else:
                v = 0
                for x in vals: v |= x
                v = 1 - v
            res.append(v)
        return res
    return per_bit

def generate_bit_manipulation_problem(rng, n_examples=8):
    fn = _random_8bit_function(rng)
    rand_bits = lambda: [rng.randint(0,1) for _ in range(8)]
    inputs = [rand_bits() for _ in range(n_examples + 1)]
    examples, query = inputs[:n_examples], inputs[-1]
    lines = ["In Alice's Wonderland, a secret bit manipulation rule transforms 8-bit binary numbers. "
             "The transformation involves operations like bit shifts, rotations, XOR, AND, OR, NOT, "
             "and possibly majority or choice functions.", "", "Here are some examples of input -> output:"]
    for b in examples:
        lines.append(f"{''.join(map(str,b))} -> {''.join(map(str,fn(b)))}")
    lines.append("")
    lines.append(f"Now, determine the output for: {''.join(map(str,query))}")
    return "\n".join(lines), "".join(map(str, fn(query)))

_SYM = list("!@#$%^&*()[]{}|<>?/\\'\"`~;:,.+-=")
def _random_symbol_string(rng, length):
    return "".join(rng.choice(_SYM) for _ in range(length))

def generate_equation_problem(rng, n_examples=4):
    chars = list(set(_SYM)); rng.shuffle(chars)
    cipher = {}
    half = len(chars) // 2
    for i, c in enumerate(chars[:half]):
        cipher[c] = chars[i + half]
    while len(cipher) < 10:
        c = rng.choice(_SYM)
        if c not in cipher:
            tgt = rng.choice(_SYM)
            if tgt not in cipher.values():
                cipher[c] = tgt
    apply = lambda s: "".join(cipher.get(c, c) for c in s)
    L = rng.randint(3, 6)
    inputs = [_random_symbol_string(rng, L) for _ in range(n_examples + 1)]
    query = inputs[-1]
    lines = ["In Alice's Wonderland, a secret set of transformation rules is applied to equations. "
             "Below are a few examples:"]
    for inp in inputs[:n_examples]:
        lines.append(f"{inp} = {apply(inp)}")
    lines.append(f"Now, determine the result for: {query}")
    return "\n".join(lines), apply(query)

GENERATORS = {
    "bit_manipulation": generate_bit_manipulation_problem,
    "cipher":           generate_cipher_problem,
    "numeral":          generate_numeral_problem,
    "unit_conversion":  generate_unit_conversion_problem,
    "gravity":          generate_gravity_problem,
    "equation":         generate_equation_problem,
}

def generate_synthetic_problems(n_per_category=500, seed=1337, categories=None):
    cats = categories or list(GENERATORS.keys())
    rng = random.Random(seed)
    out = []
    for cat in cats:
        gen = GENERATORS[cat]
        seen = set(); n = 0; attempts = 0
        while n < n_per_category and attempts < n_per_category * 6:
            attempts += 1
            try:
                prompt, answer = gen(rng)
            except Exception:
                continue
            if (prompt, answer) in seen:
                continue
            seen.add((prompt, answer))
            out.append({"prompt": prompt, "answer": answer, "category": cat, "source": "synthetic"})
            n += 1
        print(f"  {cat:18s} -> {n}")
    rng.shuffle(out)
    return out

# verifier-label map (category -> type used by verify_answer)
CATEGORY_TO_LABEL = {
    "bit_manipulation": "Bit Manipulation",
    "cipher":           "Text Encryption",
    "numeral":          "Number Base Conversion",
    "unit_conversion":  "Unit Conversion",
    "gravity":          "Gravitational Constant",
    "equation":         "Equation Transformation",
}
print("Generators ready:", list(GENERATORS.keys()))


## Verifiers + puzzle classifier (RLVR)

Brace-balanced `\boxed{}` extraction + type-aware verify (numeric tolerance for
gravity/unit, multi-base int for bit/numeral, whitespace-insensitive for
cipher/equation). `classify_label` is a regex puzzle-type detector — needed for
`train.csv` rows (no category column) so the verifier picks the right comparator.
The model never sees the category; this is server-side only.

In [ ]:
import re

_BOXED_RE = re.compile(r"\\boxed\{([^}]*)\}")
_THINK_RE = re.compile(r"<think>.*?</think>", re.DOTALL)

def _content(c):
    if isinstance(c, list) and c and isinstance(c[0], dict):
        return c[-1].get("content", "")
    return str(c)

def extract_boxed(text):
    idx = text.find("\\boxed{")
    if idx == -1:
        m = _BOXED_RE.search(text)
        return m.group(1).strip() if m else None
    depth, start = 1, idx + 7
    for i in range(start, len(text)):
        if text[i] == "{":   depth += 1
        elif text[i] == "}": depth -= 1
        if depth == 0:
            return text[start:i].strip()
    return text[start:].strip()

def _norm(s):
    return str(s).strip().lower().replace(" ", "").replace(",", "")

def _numeric_match(p, e, rel_tol=1e-2, abs_tol=1e-4):
    try:
        pf = float(str(p).replace(",", "").strip())
        ef = float(str(e).replace(",", "").strip())
    except (ValueError, TypeError):
        return False
    return abs(pf - ef) <= max(rel_tol * max(1.0, abs(ef)), abs_tol)

def _integer_match(p, e, bases=(2, 8, 10, 16)):
    ps, es = _norm(p), _norm(e)
    def parse(s, b):
        try:
            if b == 16 and s.startswith("0x"): s = s[2:]
            elif b == 2 and s.startswith("0b"): s = s[2:]
            return int(s, b)
        except ValueError:
            return None
    for b1 in bases:
        pv = parse(ps, b1)
        if pv is None:
            continue
        for b2 in bases:
            ev = parse(es, b2)
            if ev is not None and pv == ev:
                return True
    return False

def verify_answer(predicted, expected, label):
    if predicted is None:
        return False
    pn, en = _norm(predicted), _norm(expected)
    if pn == en:
        return True
    if label in ("Gravitational Constant", "Unit Conversion"):
        return _numeric_match(predicted, expected)
    if label in ("Bit Manipulation", "Number Base Conversion"):
        return _integer_match(predicted, expected)
    if label == "Text Encryption":
        return pn.replace("'", "") == en.replace("'", "")
    if label == "Equation Transformation":
        return _numeric_match(predicted, expected) or pn == en
    return _numeric_match(predicted, expected)

# ── puzzle-type classifier (for train.csv rows; order matters) ────────────────
_LABEL_PATTERNS = [
    ("Gravitational Constant",  re.compile(r"gravit|falling distance|free.?fall|0\.5\s*\*?\s*g\s*\*?\s*t", re.I)),
    ("Number Base Conversion",  re.compile(r"numeral system|number.*convert|base[- ]?\d|radix", re.I)),
    ("Unit Conversion",         re.compile(r"unit conversion|m becomes|measurement", re.I)),
    ("Text Encryption",         re.compile(r"encrypt|decrypt|cipher", re.I)),
    ("Bit Manipulation",        re.compile(r"bit manipulation|8.?bit|binary number|bitwise", re.I)),
    ("Equation Transformation", re.compile(r"transformation rule|set of transformation|determine the result", re.I)),
]
def classify_label(prompt):
    for lab, pat in _LABEL_PATTERNS:
        if pat.search(prompt or ""):
            return lab
    return "Unknown"

# smoke tests
assert verify_answer("9.81", "9.80", "Gravitational Constant")
assert verify_answer("00101010", "42", "Bit Manipulation")
assert verify_answer("HELLO", "hello", "Text Encryption")
assert not verify_answer("9.81", "10.0", "Gravitational Constant")
print("Verifier smoke tests pass.")


In [ ]:
# Per-category system prompts — only used if USE_CATEGORY_SYSTEM_PROMPT=1.
# The eval-matching default does NOT use these (eval has no category).
SYS_GENERIC = ("Solve the puzzle. Find the rule from the examples, apply it to the query, "
    "reason step by step, then put the final answer in \\boxed{}.")
_SYS = {
    "bit_manipulation": "Bit puzzle: each output bit is a boolean function of the 8 input bits.",
    "cipher": "Substitution cipher: build a char map from examples, decrypt the query.",
    "numeral": "Roman numeral conversion.",
    "unit_conversion": "Linear unit scaling: ratio=out/in, average, apply, round 2dp.",
    "gravity": "d=0.5*g*t^2 with secret g=2d/t^2; average then apply.",
    "equation": "Symbolic transformation: find the consistent rule, apply to the query.",
}
def get_system_prompt(category):
    return _SYS.get(category, SYS_GENERIC)
print("(category system prompts available; unused unless USE_CATEGORY_SYSTEM_PROMPT=1)")


## Build the GRPO dataset

Loads `DATA_SOURCE` (train.csv and/or synthetic), assigns a verifier `label`
(known category for synthetic, `classify_label` for train.csv), and builds the
**eval-matching** prompt: bare problem + `EVAL_INSTRUCTION`, no per-category
system prompt. Side columns (`raw_prompt`, `ground_truth`, `label`,
`user_with_hint_src`) feed the reward fn + the SDPG teacher.

In [ ]:
import pandas as pd
from datasets import Dataset as HFDataset

# 1) collect raw records: {prompt, answer, category(optional), label(optional)}
records = []
if DATA_SOURCE in ("train_csv", "mixed"):
    df = pd.read_csv(TRAIN_CSV_PATH)
    if not {"prompt", "answer"}.issubset(df.columns):
        raise ValueError(f"train.csv needs prompt+answer; got {list(df.columns)}")
    df = df.dropna(subset=["prompt", "answer"])
    if MAX_TRAIN_ROWS:
        df = df.sample(min(MAX_TRAIN_ROWS, len(df)), random_state=SEED)
    for r in df.itertuples(index=False):
        records.append({"prompt": str(r.prompt).replace("\r\n", "\n"),
                        "answer": str(r.answer), "source": "train_csv"})
    print(f"train.csv rows: {len(records)}")
if DATA_SOURCE in ("synthetic", "mixed"):
    syn = generate_synthetic_problems(n_per_category=SYNTH_PER_CAT, seed=SEED)
    for s in syn:
        s["label"] = CATEGORY_TO_LABEL[s["category"]]
    records += syn
    print(f"+ synthetic -> total {len(records)}")

import random as _r
_r.Random(SEED).shuffle(records)

# 2) eval-matching prompt builder (NO per-category system prompt by default)
def build_row(ex):
    label = ex.get("label") or classify_label(ex["prompt"])
    user = ex["prompt"] + EVAL_INSTRUCTION
    sysp = ""
    if USE_CATEGORY_SYSTEM_PROMPT:
        sysp = get_system_prompt(ex.get("category", "unknown"))   # (only if you opt in)
    elif GENERIC_SYSTEM_PROMPT:
        sysp = GENERIC_SYSTEM_PROMPT
    msgs = ([{"role": "system", "content": sysp}] if sysp else []) + \
           [{"role": "user", "content": user}]
    prompt_text = tokenizer.apply_chat_template(
        msgs, tokenize=False, add_generation_prompt=True)
    return {
        "prompt":             prompt_text,
        "raw_prompt":         ex["prompt"],
        "ground_truth":       ex["answer"],
        "answer":             ex["answer"],
        "label":              label,
        "system_prompt":      sysp,
        "user_with_hint_src": user,
    }

# 3) keep only answers that round-trip through the boxed extractor (same parser
#    used at reward + eval) — drops unverifiable targets (e.g. answers with { } \\)
def _roundtrips(ans):
    return extract_boxed("\\boxed{" + str(ans) + "}") == str(ans)

kept = [e for e in records if _roundtrips(e["answer"])]
print(f"Round-trip filter: kept {len(kept)} / {len(records)}")

grpo_dataset = HFDataset.from_list([build_row(e) for e in kept])

# label distribution sanity
from collections import Counter
print("label dist:", Counter(grpo_dataset["label"]))
print("\nExample prompt (truncated):\n", grpo_dataset[0]["prompt"][:500])


## Reward functions (RLVR)

`reward = format + verifiable_accuracy + overlong_penalty`
- format: `+0.5` any `\boxed{}`, `+0.3` if `<think>…</think>` closes before it.
- accuracy: `+2.0` if the type-aware verifier accepts the boxed answer.
- overlong: DAPO soft `[0,-1]` past `max_completion+buffer`.

In [ ]:
GRPO_MAX_COMPLETION = 2048   # was 8192 -> the GPU/time hog; Wonderland CoT fits 2048
GRPO_MAX_PROMPT     = 1024   # was 3072; prompts ~300-500 tok
OVERLONG_BUFFER     = 512

_per_type_stats = {}

def format_reward(completions, **kwargs):
    out = []
    for c in completions:
        t = _content(c)
        has_boxed = bool(_BOXED_RE.search(t))
        has_think = bool(_THINK_RE.search(t))
        think_before = has_think and has_boxed and t.find("</think>") < t.rfind("\\boxed{")
        out.append((0.5 if has_boxed else 0.0) + (0.3 if think_before else 0.0))
    return out

def accuracy_reward(completions, ground_truth=None, label=None, **kwargs):
    gts = ground_truth if ground_truth is not None else kwargs.get("answer")
    labels = label if label is not None else [None] * len(completions)
    out = []
    for c, gt, lab in zip(completions, gts, labels):
        pred = extract_boxed(_content(c))
        lab = lab or "Unknown"
        ok = verify_answer(pred, gt, lab)
        out.append(2.0 if ok else 0.0)
        s = _per_type_stats.setdefault(lab, {"n": 0, "correct": 0})
        s["n"] += 1; s["correct"] += int(ok)
    return out

def overlong_penalty(completions, **kwargs):
    soft = GRPO_MAX_COMPLETION + OVERLONG_BUFFER
    hard = GRPO_MAX_COMPLETION * 2
    out = []
    for c in completions:
        n = len(_content(c)) / 4.0
        if   n <= soft: out.append(0.0)
        elif n >= hard: out.append(-1.0)
        else:           out.append(-((n - soft) / (hard - soft)))
    return out

def combined_reward(completions, ground_truth=None, label=None, **kwargs):
    f = format_reward(completions)
    a = accuracy_reward(completions, ground_truth=ground_truth, label=label, **kwargs)
    o = overlong_penalty(completions)
    return [x + y + z for x, y, z in zip(f, a, o)]

_t = ["<think>\nx\n</think>\n\\boxed{42}", "no think \\boxed{42}", "<think>\ny\n</think>\n\\boxed{7}"]
print("combined_reward:", combined_reward(_t, ground_truth=["42","42","42"],
      label=["Number Base Conversion"]*3))
_per_type_stats.clear()
print("Reward functions ready.")


## SDPG self-distillation trainer

`SDPGTrainer(GRPOTrainer)` keeps Dr.GRPO and **adds** the self-distillation
term. Each step: build a privileged **teacher prompt** per sample (same base
prompt as the student + the SDPG answer hint, re-tokenized left-padded; TRL's
RepeatSampler keeps the scored batch aligned 1:1 with the rows), then run the
**same policy** over the generated completion tokens as teacher `π_θ(·|c,x)`
(no-grad) and student `π_θ(·|x)` (grad), and add the full-vocab reverse-KL
`Σ_v p_s(log p_s − log p_t)` (chunked over time). Falls back to plain Dr.GRPO if
the teacher prompts can't be built.

In [ ]:
import torch
import torch.nn.functional as F

def make_sdpg_trainer_cls(GRPOTrainerBase):
    class SDPGTrainer(GRPOTrainerBase):
        def __init__(self, *args, sdpg_beta=1e-3, distill_max_tokens=512,
                     distill_chunk=128, **kwargs):
            super().__init__(*args, **kwargs)
            self.sdpg_beta = sdpg_beta
            self.distill_max_tokens = distill_max_tokens
            self.distill_chunk = distill_chunk
            self._sdpg_log = {}

        def _build_teacher_prompts(self, inputs):
            tok = getattr(self, "processing_class", None) or self.tokenizer
            texts = []
            for ex in inputs:
                gt   = ex.get("ground_truth", ex.get("answer", ""))
                sysp = ex.get("system_prompt", "")
                user = ex.get("user_with_hint_src", ex.get("raw_prompt", ""))
                hint = (f"\nThe correct answer to this problem is: {gt}\n"
                        "Use this to verify your reasoning, but show your full solution process.")
                msgs = ([{"role": "system", "content": sysp}] if sysp else []) + \
                       [{"role": "user", "content": user + hint}]
                texts.append(tok.apply_chat_template(
                    msgs, tokenize=False, add_generation_prompt=True))
            old = tok.padding_side
            tok.padding_side = "left"
            enc = tok(texts, return_tensors="pt", padding=True, add_special_tokens=False)
            tok.padding_side = old
            dev = self.accelerator.device
            return enc["input_ids"].to(dev), enc["attention_mask"].to(dev)

        def _generate_and_score_completions(self, inputs):
            out = super()._generate_and_score_completions(inputs)
            if self.sdpg_beta > 0:
                try:
                    tpi, tpm = self._build_teacher_prompts(inputs)
                    if tpi.size(0) == out["completion_ids"].size(0):
                        out["teacher_prompt_ids"] = tpi
                        out["teacher_prompt_mask"] = tpm
                    else:
                        print(f"[SDPG] batch mismatch teacher={tpi.size(0)} "
                              f"comp={out['completion_ids'].size(0)} -> distill off this step")
                except Exception as e:
                    print("[SDPG] teacher build failed -> distill off this step:", repr(e))
            return out

        def _distill_kl(self, model, inputs):
            comp_ids  = inputs["completion_ids"]
            comp_mask = inputs["completion_mask"]
            p_ids,  p_mask  = inputs["prompt_ids"],         inputs["prompt_mask"]
            tp_ids, tp_mask = inputs["teacher_prompt_ids"], inputs["teacher_prompt_mask"]

            T = min(comp_ids.size(1), self.distill_max_tokens)
            cids, cmask = comp_ids[:, :T], comp_mask[:, :T]
            Pp, Pt = p_ids.size(1), tp_ids.size(1)

            s_ids  = torch.cat([p_ids,  cids], dim=1)
            s_mask = torch.cat([p_mask, cmask], dim=1)
            t_ids  = torch.cat([tp_ids, cids], dim=1)
            t_mask = torch.cat([tp_mask, cmask], dim=1)

            with torch.no_grad():
                t_logits = model(input_ids=t_ids, attention_mask=t_mask).logits[:, Pt-1:Pt-1+T, :]
            s_logits = model(input_ids=s_ids, attention_mask=s_mask).logits[:, Pp-1:Pp-1+T, :]

            n_tok = cmask.sum().clamp(min=1.0)
            kl_sum = s_logits.new_zeros(())
            for c0 in range(0, T, self.distill_chunk):
                c1 = min(c0 + self.distill_chunk, T)
                s_lp = F.log_softmax(s_logits[:, c0:c1, :].float(), dim=-1)
                with torch.no_grad():
                    t_lp = F.log_softmax(t_logits[:, c0:c1, :].float(), dim=-1)
                kl = (s_lp.exp() * (s_lp - t_lp)).sum(dim=-1)        # reverse KL student||teacher
                kl_sum = kl_sum + (kl * cmask[:, c0:c1]).sum()
            return kl_sum / n_tok

        def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
            base = super().compute_loss(model, inputs, return_outputs=return_outputs,
                                        num_items_in_batch=num_items_in_batch)
            base_loss = base[0] if isinstance(base, tuple) else base
            if self.sdpg_beta <= 0 or "teacher_prompt_ids" not in inputs:
                return base
            try:
                kl = self._distill_kl(model, inputs)
                loss = base_loss + self.sdpg_beta * kl
                self._sdpg_log["sdpg/distill_kl"] = float(kl.detach())
            except Exception as e:
                print("[SDPG] distill failed -> base loss only:", repr(e))
                return base
            return (loss,) + tuple(base[1:]) if isinstance(base, tuple) else loss

        def log(self, logs, *args, **kwargs):
            if getattr(self, "_sdpg_log", None):
                logs = {**logs, **self._sdpg_log}
            return super().log(logs, *args, **kwargs)

    return SDPGTrainer

print("SDPGTrainer factory ready.")


## Dr.GRPO config

Purges Unsloth's patched GRPO trainer (incompatible with Nemotron-H) and
re-imports vanilla TRL. `beta = SDPG_ALPHA_REF` keeps a small reference-KL
anchor to the SFT (protects the 85%).

In [ ]:
import inspect, gc

os.environ["TORCHDYNAMO_DISABLE"]   = "1"
os.environ["TORCH_COMPILE_DISABLE"] = "1"
import torch, torch._dynamo
torch._dynamo.config.disable = True
torch._dynamo.reset()

for _m in list(sys.modules):
    if _m == "trl" or _m.startswith("trl.") or "unsloth" in _m.lower():
        del sys.modules[_m]
sys.meta_path = [f for f in sys.meta_path if "unsloth" not in type(f).__module__.lower()]

from trl import GRPOTrainer, GRPOConfig
assert "unsloth" not in GRPOTrainer.__module__.lower(), GRPOTrainer.__module__
print("GRPOTrainer module:", GRPOTrainer.__module__)

_p = set(inspect.signature(GRPOConfig.__init__).parameters)
grpo_kwargs = dict(
    output_dir                   = os.path.join(OUTPUT_ROOT, "grpo_run"),
    num_train_epochs             = 1,
    per_device_train_batch_size  = 1,
    gradient_accumulation_steps  = 2,
    learning_rate                = 1e-6,
    lr_scheduler_type            = "constant",
    warmup_ratio                 = 0.0,
    weight_decay                 = 0.0,
    max_grad_norm                = 1.0,
    max_prompt_length            = GRPO_MAX_PROMPT,
    max_completion_length        = GRPO_MAX_COMPLETION,
    num_generations              = 4,   # was 2; more group-relative signal (mem freed by beta=0 + shorter completion)
    temperature                  = 1.0,
    top_p                        = 1.0,
    beta                         = SDPG_ALPHA_REF,   # reference-policy KL (anchor to SFT)
    optim                        = "paged_adamw_8bit",
    logging_steps                = 1,
    logging_dir                  = TB_LOG_DIR,
    report_to                    = "tensorboard",
    save_strategy                = "no",
    bf16                         = True,
    gradient_checkpointing       = True,
    gradient_checkpointing_kwargs= {"use_reentrant": True},
    seed                         = SEED,
    remove_unused_columns        = False,
)
if "scale_rewards" in _p:
    grpo_kwargs["scale_rewards"] = False
if "loss_type" in _p:
    try:
        GRPOConfig(loss_type="dr_grpo"); grpo_kwargs["loss_type"] = "dr_grpo"
    except Exception:
        grpo_kwargs["loss_type"] = "dapo"
if "mask_truncated_completions" in _p:
    grpo_kwargs["mask_truncated_completions"] = True
if "epsilon_high" in _p:
    grpo_kwargs["epsilon"] = 0.2
    grpo_kwargs["epsilon_high"] = 0.28

grpo_config = GRPOConfig(**grpo_kwargs)
print("loss_type:", getattr(grpo_config, "loss_type", "n/a"),
      "| scale_rewards:", getattr(grpo_config, "scale_rewards", "n/a"),
      "| beta(ref-KL):", grpo_config.beta, "| num_gen:", grpo_config.num_generations)


In [ ]:
from transformers import TrainerCallback

class PerTypeAccuracyCallback(TrainerCallback):
    def __init__(self, stats):
        self.stats = stats
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is None:
            return
        for ptype, s in self.stats.items():
            if s["n"] > 0:
                key = ptype.replace(" ", "_").lower()
                logs[f"acc/{key}"]   = s["correct"] / s["n"]
                logs[f"count/{key}"] = s["n"]
        for s in self.stats.values():
            s["n"] = 0; s["correct"] = 0

print("Callback ready.")


## Launch training

Pre-flight greedy-decodes one prompt to confirm `\boxed{}` is emitted (CLAUDE.md:
sanity-check before multi-hour runs).

In [ ]:
import gc, time, torch

common = dict(
    model            = model,
    args             = grpo_config,
    train_dataset    = grpo_dataset,
    reward_funcs     = [combined_reward],
    processing_class = tokenizer,
    callbacks        = [PerTypeAccuracyCallback(_per_type_stats)],
)

if USE_SDPG:
    SDPGTrainer = make_sdpg_trainer_cls(GRPOTrainer)
    trainer = SDPGTrainer(sdpg_beta=SDPG_BETA, distill_max_tokens=SDPG_DISTILL_MAX_T,
                          distill_chunk=SDPG_DISTILL_CHUNK, **common)
    print(f"Trainer: SDPGTrainer (beta={SDPG_BETA}, ref-KL alpha={SDPG_ALPHA_REF})")
else:
    trainer = GRPOTrainer(**common)
    print("Trainer: GRPOTrainer (plain Dr.GRPO)")

torch.cuda.empty_cache(); gc.collect()

print("Pre-flight: greedy-decode one prompt...")
try:
    _ids = tokenizer(grpo_dataset[0]["prompt"], return_tensors="pt").to(model.device)
    with torch.no_grad():
        _out = model.generate(**_ids, max_new_tokens=256, do_sample=False)
    _txt = tokenizer.decode(_out[0][_ids["input_ids"].shape[1]:], skip_special_tokens=True)
    print("  tail:", repr(_txt[-160:]))
    print("  boxed:", extract_boxed(_txt))
except Exception as e:
    print("  pre-flight skipped:", repr(e))

t0 = time.time()
trainer.train()
print(f"Training done in {(time.time()-t0)/60:.1f} min")

os.makedirs(GRPO_ADAPTER_DIR, exist_ok=True)
model.save_pretrained(GRPO_ADAPTER_DIR)
tokenizer.save_pretrained(GRPO_ADAPTER_DIR)
print("Adapter saved ->", GRPO_ADAPTER_DIR)


## Package `submission.zip` (LoRA only)

**Eval-gate first:** only ship this if it beats the 85% SFT on a held-out set —
otherwise keep the SFT adapter (best-of-2). RL can regress a strong SFT.

In [ ]:
import json, shutil, zipfile

required = ["adapter_config.json", "adapter_model.safetensors"]
for f in required:
    sp = os.path.join(GRPO_ADAPTER_DIR, f)
    if not os.path.exists(sp):
        raise FileNotFoundError(f"Missing {sp}")
    shutil.copy2(sp, os.path.join(SUBMISSION_DIR, f))
    print(f"  copied {f}  ({os.path.getsize(os.path.join(SUBMISSION_DIR, f))/1024/1024:.1f} MB)")

cfg_path = os.path.join(SUBMISSION_DIR, "adapter_config.json")
with open(cfg_path) as f:
    cfg = json.load(f)
cfg["base_model_name_or_path"] = BASE_MODEL_NAME
cfg["inference_mode"] = True
cfg["lora_dropout"]   = 0.0
assert cfg.get("r", MAX_LORA_RANK) <= MAX_LORA_RANK, "max_lora_rank > 32 will be rejected"
with open(cfg_path, "w") as f:
    json.dump(cfg, f, indent=2)

zip_path = os.path.join(OUTPUT_ROOT, "submission.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in required:
        zf.write(os.path.join(SUBMISSION_DIR, f), f)
print(f"\nsubmission.zip -> {zip_path}  ({os.path.getsize(zip_path)/1024/1024:.1f} MB)")


---
**Resume-on-85% checklist**
1. Set `SFT_ADAPTER_DIR` to the 85% adapter dir. Confirm the printed rank ≤ 32.
2. `DATA_SOURCE="train_csv"`, point `TRAIN_CSV_PATH` at the real train.csv.
3. Keep `USE_CATEGORY_SYSTEM_PROMPT=0` (eval gives no category). Verify the
   pre-flight decode already boxes correctly (the SFT should).
4. Keep `beta=SDPG_ALPHA_REF=1e-3` (anchor to SFT), LR 1e-6.
5. Eval-gate the result vs the SFT before submitting; keep SFT as fallback.
6. `USE_SDPG=1` only on a big GPU; helps the hard residual where reward saturates.
